In [6]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
from dataclasses import dataclass
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from natsort import natsorted
import numpy as np
import matplotlib.animation as animation
import xarray as xr
import h5py
import imageio
import matplotlib
import gc
import sys
import io
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from scipy.optimize import curve_fit
import scipy.integrate


# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'../../../../tidy3d'))

from AutomationModule import * 

import AutomationModule as AM
plt.rcParams.update({'font.size': 6.7})  

tidy3dAPI = os.environ["API_TIDY3D_KEY"]
plt.rc('font', family='Arial')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
data_directory  = rf"./data/intensities.h5"
data_extracted = AM.read_hdf5_as_dict(data_directory)

In [16]:
data_extracted.keys()

dict_keys(['chi', 'data_field_intensities', 'f', 'sample', 'size', 'x', 'y'])

In [19]:
def adjustFigAspect(fig,aspect=1):
    '''
    Adjust the subplot parameters so that the figure has the correct
    aspect ratio.
    '''
    xsize,ysize = fig.get_size_inches()
    minsize = min(xsize,ysize)
    xlim = .4*minsize/xsize
    ylim = .4*minsize/ysize
    if aspect < 1:
        xlim *= aspect
    else:
        ylim /= aspect
    fig.subplots_adjust(left=.5-xlim,
                        right=.5+xlim,
                        bottom=.5-ylim,
                        top=.5+ylim)
    

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import ipywidgets as widgets
from IPython.display import display


def create_map(field_time_out, x, y, f, monitor_lambdas, directory_patches,
               t=None, type='f', log=False, normalize=True, a=1.0,
               overlay=True, overlay_sample=0,
               absorber_thickness=6.0, absorber_axis='y',
               save_dir="./figures/intensity_field_maps"):
    """Interactively browse the sample-averaged intensity field maps.

    field_time_out has shape (chi, sizes, samples, x, y, freqs); the displayed
    map is the ensemble average over all `samples`.

    Controls:
      * chi / size dropdowns  -> pick the (chi, L) cube
      * frame slider          -> pick the frequency (nu) / time frame (labelled
                                 with the actual nu value)
      * structure overlay      -> draw one realization's scatterer outlines
      * Save frame            -> writes the current frame to
            {save_dir}/freq__{nu[i]}_chi_{chi}_size_{L}.pdf
        at figsize=(3.5, 2), dpi=300, Arial size 8.

    absorber_thickness : float
        Width (in units of a) of the gray gradient band drawn at each end to
        mark the absorbing boundaries. Set to 0 to disable.
    absorber_axis : {'x', 'y', 'both'}
        Which pair of ends gets the bands. 'x' = top & bottom (vertical, the
        propagation direction for these periodic beam-spreading runs), 'y' =
        left & right, 'both' = all four.

    Figures are built with a standalone Agg canvas (no pyplot global state), so
    the inline backend cannot duplicate them.
    """
    RC = {'font.family': 'Arial', 'font.size': 8}

    def _fmt(v):
        """Match how the values appear in the folder names: drop the trailing
        .0 on integral floats (Sample_0.0 -> Sample_0, L=100.0 -> L=100)."""
        if isinstance(v, float) and v.is_integer():
            return str(int(v))
        return str(v)

    chi_values = list(directory_patches["chi"])
    size_values = list(directory_patches["L"])
    sample_values = list(directory_patches["sample"])

    xa = x / a
    ya = y / a

    # normalized frequency nu = f / C_0 = 1 / lambda
    nu = np.asarray(f) / td.C_0
    n_frames = field_time_out.shape[-1]

    # --- custom colormap: White -> Blue -> Yellow -> Green -> Red -> Black ---
    colors = [
        (1, 1, 1),  # White
        (0, 0, 1),  # Blue
        (1, 1, 0),  # Yellow
        (0, 1, 0),  # Green
        (1, 0, 0),  # Red
        (0, 0, 0),  # Black
    ]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_colormap", colors, N=500)

    def add_absorber(ax, extent, direction, gray=0.5, max_alpha=0.7, n=256):
        """Draw a solid-gray gradient band, opaque at the outer edge (`direction`)
        and fading to transparent toward the interior."""
        if direction in ('up', 'down'):
            grad = np.linspace(0.0, max_alpha, n)          # bottom -> top
            if direction == 'down':
                grad = grad[::-1]
            rgba = np.zeros((n, 1, 4))
            rgba[:, 0, 3] = grad
        else:
            grad = np.linspace(0.0, max_alpha, n)          # left -> right
            if direction == 'left':
                grad = grad[::-1]
            rgba = np.zeros((1, n, 4))
            rgba[0, :, 3] = grad
        rgba[..., 0:3] = gray
        ax.imshow(rgba, extent=extent, origin='lower', aspect='auto',
                  interpolation='bilinear', zorder=5)

    # --- lazy, cached loaders (avoid recomputing/refetching on every redraw) ---
    _field_cache = {}
    _patch_cache = {}

    def get_field_disp(ci, si):
        key = (ci, si)
        if key not in _field_cache:
            fld = field_time_out[ci, si, :, :, :, :].mean(axis=0)   # avg over samples
            mx = fld.max(axis=(0, 1)) if normalize else 1
            fld = fld / mx
            _field_cache[key] = np.log10(fld) if log else fld
        return _field_cache[key]

    def get_patches(ci, si):
        key = (ci, si)
        if key not in _patch_cache:
            chi_val = chi_values[ci]
            L_str = _fmt(size_values[si])
            sample_str = _fmt(sample_values[overlay_sample])
            try:
                # the ".txt" is a directory holding a local tidy3d bundle, so
                # from_hdf5=True reads "<...>.txt/Data.hdf5" directly (no web, no
                # open() on the directory).
                sim_data = AM.loadFromFile(
                    key=tidy3dAPI,
                    file_path=rf"{directory_patches["path"]}/chi_{chi_val:.2f}_N_10000_posics/chi_{chi_val:.2f}_N_10000_posics_L={L_str} nu=0.2 - 0.48/chi_{chi_val:.2f}_N_10000_posics Beam Spreading 0.2 - 0.48 - Sample_{sample_str} 50.0ps L={L_str}a.txt",
                    get_ref=False, verbose=False, from_hdf5=True,
                ).sim_data
                fig1 = Figure()
                FigureCanvasAgg(fig1)
                ax_1 = fig1.add_subplot(111)
                ax1 = sim_data.simulation.plot_structures_eps(freq=monitor_lambdas[0], cbar=False,
                                                              z=0, ax=ax_1, reverse=False)
                _patch_cache[key] = [p.get_path() for p in ax1.patches]
            except Exception as e:
                print(f"[patches] could not load structure for chi={chi_val:.2f}, L={L_str}: {e}")
                _patch_cache[key] = []
        return _patch_cache[key]

    def make_fig(ci, si, i, draw_patches):
        """Render frame i for cube (ci, si) as a standalone (non-pyplot) Figure."""
        field_disp = get_field_disp(ci, si)
        with plt.rc_context(RC):
            fig = Figure(figsize=(3.5, 4), dpi=300)
            FigureCanvasAgg(fig)
            ax = fig.add_subplot(111)
            im = ax.imshow(field_disp[:, :, i],
                           vmin=np.min(field_disp[:, :, i]), vmax=np.max(field_disp[:, :, i]),
                           extent=[np.min(ya), np.max(ya), np.min(xa), np.max(xa)],
                           interpolation='gaussian', origin='lower', cmap=cmap, aspect='equal')
            if type == "t":
                ax.set_title(f'Time: {t[i]} ps')
            else:
                ax.set_title(f'$\\nu$: {nu[i]:.4g}')

            cbar = fig.colorbar(im, ax=ax, orientation='vertical',
                                fraction=0.1, pad=0.1, shrink=0.3)
            cbar.set_label(rf"$|E|^2$ Normalized")
            cbar.ax.yaxis.set_label_position('left')
            ax.set_ylabel(rf"x(a)")
            ax.set_xlabel(rf"y(a)")
            ax.set_xlim(-25, 25)
            ax.tick_params(which='major')

            if draw_patches:
                for path_patch in get_patches(ci, si):
                    new_patch = patches.PathPatch(path_patch,
                                                  edgecolor=(0, 0, 0, 0.55), facecolor="none")
                    new_patch.set_transform(
                        matplotlib.transforms.Affine2D().rotate_deg(90) + ax.transData)
                    ax.add_patch(new_patch)

            # --- gray gradient bands marking the absorbing boundaries ---
            if absorber_thickness and absorber_thickness > 0:
                th = absorber_thickness
                xlo, xhi = ax.get_xlim()           # horizontal (y-axis) span shown
                ylo, yhi = np.min(xa), np.max(xa)  # vertical (x-axis) data span
                if absorber_axis in ('x', 'both'):
                    add_absorber(ax, [xlo, xhi, yhi - th, yhi], 'up')     # top
                    add_absorber(ax, [xlo, xhi, ylo, ylo + th], 'down')   # bottom
                if absorber_axis in ('y', 'both'):
                    add_absorber(ax, [xhi - th, xhi, ylo, yhi], 'right')  # right
                    add_absorber(ax, [xlo, xlo + th, ylo, yhi], 'left')   # left
                ax.set_xlim(xlo, xhi)
                ax.set_ylim(ylo, yhi)

            ax.set_aspect('auto', adjustable='box')
        return fig

    def fig_to_png(fig):
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        return buf.getvalue()

    # --- widgets ---
    chi_dd = widgets.Dropdown(options=[(f"{v:.2f}", k) for k, v in enumerate(chi_values)],
                              value=0, description="chi")
    size_dd = widgets.Dropdown(options=[(_fmt(v), k) for k, v in enumerate(size_values)],
                               value=0, description="size (L)")
    # slider labelled with the actual nu value (option label = nu, value = frame index)
    slider_desc = "$\\nu$" if type != "t" else "t (ps)"
    slider_options = ([(f"{nu[i]:.4g}", i) for i in range(n_frames)] if type != "t"
                      else [(f"{t[i]:.4g}", i) for i in range(n_frames)])
    frame_slider = widgets.SelectionSlider(options=slider_options, value=0,
                                           description=slider_desc, continuous_update=False,
                                           style={'description_width': 'initial'},
                                           layout=widgets.Layout(width='500px'))
    overlay_chk = widgets.Checkbox(value=overlay, description="structure overlay")
    save_button = widgets.Button(description="Save frame", icon='save', button_style='primary')
    status = widgets.Label(value="")
    img = widgets.Image(format='png', layout=widgets.Layout(width='525px'))

    def render(*_):
        fig = make_fig(chi_dd.value, size_dd.value, frame_slider.value, overlay_chk.value)
        img.value = fig_to_png(fig)

    def on_save(_):
        ci, si, i = chi_dd.value, size_dd.value, frame_slider.value
        chi_val, L_str = chi_values[ci], _fmt(size_values[si])
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{save_dir}/freq__{nu[i]:.4g}_chi_{chi_val:.2f}_size_{L_str}.pdf"
        fig = make_fig(ci, si, i, overlay_chk.value)
        fig.savefig(fname, bbox_inches='tight')
        status.value = f"Saved  {fname}"

    for w in (chi_dd, size_dd, frame_slider, overlay_chk):
        w.observe(render, names='value')
    save_button.on_click(on_save)

    display(widgets.VBox([widgets.HBox([chi_dd, size_dd, overlay_chk]),
                          frame_slider,
                          widgets.HBox([save_button, status]),
                          img]))
    render()
    return ""


In [21]:
# --- interactive intensity map browser (averaged over samples) ---
# Pick chi and size from the dropdowns; the map is the mean over all samples.

directory_patches = {
    "path": rf"F:\2D SHU Chi Statistics Loosy Data\data\10_07_2024 Beam Spreading Broad Bandwidth Periodic Conditions Freq Domain",   # base folder holding the chi_*_N_10000_posics dirs
    "chi":    data_extracted["chi"],
    "L":      data_extracted["size"],
    "sample": data_extracted["sample"],
}

create_map(
    field_time_out=data_extracted["data_field_intensities"],
    x=data_extracted["x"],
    y=data_extracted["y"],
    f=data_extracted["f"],
    monitor_lambdas=data_extracted["f"],   # freq passed to plot_structures_eps
    directory_patches=directory_patches,
    type='f',
    log=False,
    normalize=True,
    overlay=True,          # show scatterer outlines
    overlay_sample=0,      # which realization's structure to overlay
    absorber_thickness=7.0,  # width of gray gradient band marking absorbing boundaries
    absorber_axis='y',     # which pair of ends gets the bands ('x', '
    save_dir="./figures/intensity_field_maps",
)


Configured successfully.


14:29:19 W. Europe Daylight Time WARNING: updating Simulation from 2.7 to 2.9   

''

14:29:27 W. Europe Daylight Time WARNING: updating Simulation from 2.7 to 2.9   

14:30:36 W. Europe Daylight Time WARNING: updating Simulation from 2.7 to 2.9   